# 문항 조합별 분할 데이터 생성

이 노트북은 BERT 실험을 위한 **문항(question) 단위 분할 데이터**를 생성합니다. 원본 데이터에는 A형(q1~q4)과 B형(q1~q4), 총 8개 문항이 있으며, 이를 Q1~Q8로 매핑합니다.

**목적**:
- **Part 1 (문항별 데이터)**: Q1~Q8 각각을 개별 Excel 파일로 저장합니다. 이 파일들을 다양하게 조합하여 254가지 훈련/테스트 분할을 생성할 수 있습니다.
- **Part 2 (학생 기반 분할)**: 55명의 학생을 비율별(80:20~40:60)로 분리하여, 학생이 완전히 독립된 훈련/테스트 세트를 생성합니다.

**출력 파일**:
- `BERT_split/question_data/` : Q1~Q8.xlsx + question_mapping.json
- `BERT_split/student_splits/` : train/test x 15쌍 + split_info.json

## 0. 설정 및 데이터 로드

### 라이브러리 및 경로 설정

문항 매핑(Q_MAP)은 Q1~Q8을 (검사지유형, 질문번호) 쌍으로 변환합니다.
- Q1~Q4: A형 검사지의 q1~q4
- Q5~Q8: B형 검사지의 q1~q4

학생 분할은 5가지 비율 x 3가지 시드(seed) = 15개 분할을 생성합니다.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

from kiwipiepy import Kiwi  # 한국어 형태소 분석기

# === 경로 ===
DATA_PATH = Path("../data/feedback_data.xlsx")
SPLIT_DIR = Path("./BERT_split")              # BERT 실험용 분할 데이터 최상위 디렉토리
SPLIT_QC_DIR = SPLIT_DIR / "question_data"    # 문항별 데이터 저장
SPLIT_SS_DIR = SPLIT_DIR / "student_splits"   # 학생 분할 데이터 저장

SPLIT_DIR.mkdir(exist_ok=True)
SPLIT_QC_DIR.mkdir(exist_ok=True)
SPLIT_SS_DIR.mkdir(exist_ok=True)

# === 문항 매핑: Q1~Q8 → (검사지유형, 질문번호) ===
Q_MAP = {
    "Q1": ("a", "q1"), "Q2": ("a", "q2"), "Q3": ("a", "q3"), "Q4": ("a", "q4"),  # A형
    "Q5": ("b", "q1"), "Q6": ("b", "q2"), "Q7": ("b", "q3"), "Q8": ("b", "q4"),  # B형
}
ALL_QUESTIONS = list(Q_MAP.keys())

# === 학생 분할 설정 ===
# 5가지 비율: 훈련 학생 비율을 80%에서 40%까지 변화
SPLIT_RATIOS = [(80, 20), (70, 30), (60, 40), (50, 50), (40, 60)]
# 3개 시드: 같은 비율에서 서로 다른 학생 배정을 생성하여 결과의 안정성 확인
SPLIT_SEEDS = [42, 123, 456]

# 저장할 컬럼 목록 (원본 + 파싱된 메타 정보 + 토큰화 결과)
SAVE_COLS = ["id", "feedback_text", "label", "student", "form", "question", "tokenized"]

print(f"설정 완료: {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# 데이터 로드
df = pd.read_excel(DATA_PATH)
print(f"전체 데이터: {len(df)}개, 컬럼: {df.columns.tolist()}\n")

# ID 파싱: 정규표현식으로 학생/검사지/질문 정보 추출
df["student"] = df["id"].str.extract(r"^(\d+)_")      # 학생 번호
df["form"] = df["id"].str.extract(r"_([ab])q")         # 검사지 유형
df["question"] = df["id"].str.extract(r"_[ab](q\d)_")  # 질문 번호

print(f"학생: {df['student'].nunique()}명")
print(f"검사지: {sorted(df['form'].unique())}")
print(f"질문: {sorted(df['question'].unique())}")
print(f"\n=== 레이블 분포 ===")
print(df["label"].value_counts().sort_index())

In [ ]:
# 형태소 분석 (Kiwi + Lemmatization)
class KiwiLemmaTokenizer:
    # 의미 있는 품사만 추출: 명사, 동사, 형용사, 부사 계열
    INCLUDE_POS = {"NNG", "NNP", "NNB", "NR", "VV", "VA", "VX", "MAG", "MAJ"}

    def __init__(self):
        self.kiwi = Kiwi()

    def tokenize_with_lemma(self, text):
        if pd.isna(text) or str(text).strip() == "":
            return ""
        tokens = self.kiwi.tokenize(str(text))
        # 원형(lemma) 우선, 없으면 표면형(form) 사용
        return " ".join(t.lemma or t.form for t in tokens if t.tag in self.INCLUDE_POS)

print("토큰화 중...")
tokenizer = KiwiLemmaTokenizer()
df["tokenized"] = df["feedback_text"].apply(tokenizer.tokenize_with_lemma)
print(f"토큰화 완료! ({len(df)}건)")
print(f"예시: '{df['tokenized'].iloc[0]}'")

## 1. 문항별 데이터 분할 (Part 1: 문항 조합 실험용)

Q1~Q8 각 문항의 데이터를 개별 Excel 파일로 저장합니다. 이 8개 파일을 다양하게 조합하면 254가지(2^8 - 2, 전체/빈집합 제외) 훈련/테스트 분할을 구성할 수 있습니다.

예를 들어:
- 훈련: Q1+Q2+Q3, 테스트: Q4+Q5+Q6+Q7+Q8 (문항 조합)
- 훈련: Q1~Q4(A형 전체), 테스트: Q5~Q8(B형 전체) (검사지 기반)

이러한 조합 방식은 BERT 실험 코드에서 동적으로 생성됩니다.

In [ ]:
def get_question_data(df, question_label):
    """Q_MAP을 사용하여 특정 문항(Q1~Q8)의 데이터를 추출."""
    form, q = Q_MAP[question_label]  # 예: Q5 → ('b', 'q1')
    return df[(df["form"] == form) & (df["question"] == q)]

print("=== 문항별 데이터 저장 ===\n")

q_summary = {}
for q_label in ALL_QUESTIONS:
    q_data = get_question_data(df, q_label)
    # 각 문항을 개별 Excel 파일로 저장
    q_data[SAVE_COLS].to_excel(SPLIT_QC_DIR / f"{q_label}.xlsx", index=False)
    # 메타 정보를 JSON으로 기록 (나중에 조합 시 참조)
    q_summary[q_label] = {
        "form": Q_MAP[q_label][0],
        "question": Q_MAP[q_label][1],
        "n_samples": len(q_data),
        "label_dist": {str(k): v for k, v in q_data["label"].value_counts().sort_index().items()},
    }
    print(f"  {q_label} -> {q_label}.xlsx ({len(q_data)}건)")

# 문항 매핑 정보를 JSON으로 저장
with open(SPLIT_QC_DIR / "question_mapping.json", "w", encoding="utf-8") as f:
    json.dump(q_summary, f, indent=2, ensure_ascii=False)

print(f"\n  question_mapping.json 저장")
print(f"  총 {len(ALL_QUESTIONS)}개 문항 파일 완료")

## 2. 학생 기반 분할 (Part 2: 학생 독립성 검증용)

55명 학생을 비율별로 훈련/테스트에 **완전 분리**합니다. 한 학생의 모든 데이터가 훈련 또는 테스트 한쪽에만 배정됩니다.

- **5가지 비율**: 80:20, 70:30, 60:40, 50:50, 40:60
- **3가지 시드**: 42, 123, 456 (각 비율에서 서로 다른 학생 배정)
- **총 15개 분할** x 2(train/test) = **30개 파일**

학생 수가 55명으로 적기 때문에, 정확히 N%를 맞추기 어렵습니다. 예: 80%는 44명, 20%는 11명(55 x 0.8 = 44).

In [ ]:
def split_by_students(df, train_ratio, seed):
    """학생 ID 기반으로 데이터를 분할. 한 학생의 모든 데이터는 한쪽에만 배정됨."""
    student_ids = sorted(df["student"].unique())
    n_students = len(student_ids)
    n_train = int(n_students * train_ratio / 100)  # 훈련에 배정할 학생 수

    rng = np.random.RandomState(seed)       # 시드로 재현 가능한 난수 생성기
    shuffled = rng.permutation(student_ids)  # 학생 목록을 무작위로 섞기

    train_students = set(shuffled[:n_train])   # 앞부분 → 훈련
    test_students = set(shuffled[n_train:])     # 뒷부분 → 테스트

    # 학생 기준으로 데이터 분리
    train_df = df[df["student"].isin(train_students)].copy()
    test_df = df[df["student"].isin(test_students)].copy()

    return train_df, test_df, train_students, test_students

print("=== 학생 분할 데이터 저장 ===\n")

ss_summary = {}
for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f"{train_pct}:{test_pct}"
    for seed in SPLIT_SEEDS:
        key = f"{ratio_name}_seed{seed}"
        safe_name = key.replace(":", "_")  # 파일명에 콜론 사용 불가하므로 언더스코어로 대체

        train_df, test_df, train_stu, test_stu = split_by_students(df, train_pct, seed)

        # Excel 파일로 저장
        train_df[SAVE_COLS].to_excel(SPLIT_SS_DIR / f"train_{safe_name}.xlsx", index=False)
        test_df[SAVE_COLS].to_excel(SPLIT_SS_DIR / f"test_{safe_name}.xlsx", index=False)

        # 메타 정보 기록 (학생 목록, 데이터 크기, 레이블 분포)
        ss_summary[key] = {
            "train_students": sorted(train_stu),
            "test_students": sorted(test_stu),
            "train_n": len(train_df),
            "test_n": len(test_df),
            "train_label_dist": {str(k): v for k, v in train_df["label"].value_counts().sort_index().items()},
            "test_label_dist": {str(k): v for k, v in test_df["label"].value_counts().sort_index().items()},
        }

        print(f"  {key}: train({len(train_df)}건), test({len(test_df)}건)")

# 분할 정보를 JSON으로 저장
with open(SPLIT_SS_DIR / "split_info.json", "w", encoding="utf-8") as f:
    json.dump(ss_summary, f, indent=2, ensure_ascii=False)

print(f"\n  split_info.json 저장")
print(f"  총 {len(ss_summary)}개 분할 x 2 = {len(ss_summary) * 2}개 파일 완료")

## 3. 검증

생성된 파일의 무결성을 확인합니다.
1. 문항별 파일: 크기와 컬럼 구조 확인
2. 학생 분할 파일: 크기 확인
3. **학생 겹침 검증**: 같은 비율의 서로 다른 시드 간 테스트 학생의 겹침 정도를 확인합니다. 겹침이 적을수록 3개 시드가 다양한 학생 구성을 검증함을 의미합니다.

In [4]:
def get_question_data(df, question_label):
    form, q = Q_MAP[question_label]
    return df[(df["form"] == form) & (df["question"] == q)]

print("=== 문항별 데이터 저장 ===\n")

q_summary = {}
for q_label in ALL_QUESTIONS:
    q_data = get_question_data(df, q_label)
    q_data[SAVE_COLS].to_excel(SPLIT_QC_DIR / f"{q_label}.xlsx", index=False)
    q_summary[q_label] = {
        "form": Q_MAP[q_label][0],
        "question": Q_MAP[q_label][1],
        "n_samples": len(q_data),
        "label_dist": {str(k): v for k, v in q_data["label"].value_counts().sort_index().items()},
    }
    print(f"  {q_label} -> {q_label}.xlsx ({len(q_data)}건)")

with open(SPLIT_QC_DIR / "question_mapping.json", "w", encoding="utf-8") as f:
    json.dump(q_summary, f, indent=2, ensure_ascii=False)

print(f"\n  question_mapping.json 저장")
print(f"  총 {len(ALL_QUESTIONS)}개 문항 파일 완료")

=== 문항별 데이터 저장 ===

  Q1 -> Q1.xlsx (265건)
  Q2 -> Q2.xlsx (265건)
  Q3 -> Q3.xlsx (265건)
  Q4 -> Q4.xlsx (265건)
  Q5 -> Q5.xlsx (255건)
  Q6 -> Q6.xlsx (255건)
  Q7 -> Q7.xlsx (255건)
  Q8 -> Q8.xlsx (255건)

  question_mapping.json 저장
  총 8개 문항 파일 완료


**결과 해석:** 모든 파일이 정상적으로 생성되었습니다.

학생 겹침 검증 결과:
- **80:20**(테스트 11명): 시드 간 겹침 18~36%로 비교적 적음 → 다양한 학생 구성 검증 가능
- **40:60**(테스트 33명): 시드 간 겹침 54~64%로 높음 → 테스트 학생이 많아 자연스럽게 겹침 증가

이는 테스트 학생 수가 적을수록(예: 80:20) 시드 간 다양성이 높아지고, 테스트 학생 수가 많을수록(예: 40:60) 겹침이 불가피함을 보여줍니다.

## 2. 학생 기반 분할 (Part 2용)

55명 학생을 비율별로 train/test 완전 분리.  
5비율 × 3seed = 15분할 × 2(train/test) = 30개 파일.

In [5]:
def split_by_students(df, train_ratio, seed):
    student_ids = sorted(df["student"].unique())
    n_students = len(student_ids)
    n_train = int(n_students * train_ratio / 100)

    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(student_ids)

    train_students = set(shuffled[:n_train])
    test_students = set(shuffled[n_train:])

    train_df = df[df["student"].isin(train_students)].copy()
    test_df = df[df["student"].isin(test_students)].copy()

    return train_df, test_df, train_students, test_students

print("=== 학생 분할 데이터 저장 ===\n")

ss_summary = {}
for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f"{train_pct}:{test_pct}"
    for seed in SPLIT_SEEDS:
        key = f"{ratio_name}_seed{seed}"
        safe_name = key.replace(":", "_")

        train_df, test_df, train_stu, test_stu = split_by_students(df, train_pct, seed)

        train_df[SAVE_COLS].to_excel(SPLIT_SS_DIR / f"train_{safe_name}.xlsx", index=False)
        test_df[SAVE_COLS].to_excel(SPLIT_SS_DIR / f"test_{safe_name}.xlsx", index=False)

        ss_summary[key] = {
            "train_students": sorted(train_stu),
            "test_students": sorted(test_stu),
            "train_n": len(train_df),
            "test_n": len(test_df),
            "train_label_dist": {str(k): v for k, v in train_df["label"].value_counts().sort_index().items()},
            "test_label_dist": {str(k): v for k, v in test_df["label"].value_counts().sort_index().items()},
        }

        print(f"  {key}: train({len(train_df)}건), test({len(test_df)}건)")

with open(SPLIT_SS_DIR / "split_info.json", "w", encoding="utf-8") as f:
    json.dump(ss_summary, f, indent=2, ensure_ascii=False)

print(f"\n  split_info.json 저장")
print(f"  총 {len(ss_summary)}개 분할 x 2 = {len(ss_summary) * 2}개 파일 완료")

=== 학생 분할 데이터 저장 ===

  80:20_seed42: train(1680건), test(400건)
  80:20_seed123: train(1660건), test(420건)
  80:20_seed456: train(1680건), test(400건)
  70:30_seed42: train(1440건), test(640건)
  70:30_seed123: train(1420건), test(660건)
  70:30_seed456: train(1460건), test(620건)
  60:40_seed42: train(1280건), test(800건)
  60:40_seed123: train(1220건), test(860건)
  60:40_seed456: train(1260건), test(820건)
  50:50_seed42: train(1040건), test(1040건)
  50:50_seed123: train(1000건), test(1080건)
  50:50_seed456: train(1040건), test(1040건)
  40:60_seed42: train(860건), test(1220건)
  40:60_seed123: train(800건), test(1280건)
  40:60_seed456: train(860건), test(1220건)

  split_info.json 저장
  총 15개 분할 x 2 = 30개 파일 완료


## 3. 검증

In [6]:
# 생성된 파일 확인
print("=== 생성 파일 확인 ===\n")

qc_files = sorted(SPLIT_QC_DIR.glob("*.xlsx"))
ss_files = sorted(SPLIT_SS_DIR.glob("*.xlsx"))

print(f"[question_data] {len(qc_files)}개 파일")
for f in qc_files:
    tmp = pd.read_excel(f)
    print(f"  {f.name}: {len(tmp)}건, 컬럼={list(tmp.columns)}")

print(f"\n[student_splits] {len(ss_files)}개 파일")
for f in ss_files:
    tmp = pd.read_excel(f)
    print(f"  {f.name}: {len(tmp)}건")

# 학생 겹침 검증
print("\n=== 학생 분할 겹침 검증 ===")
for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f"{train_pct}:{test_pct}"
    test_sets = []
    for seed in SPLIT_SEEDS:
        key = f"{ratio_name}_seed{seed}"
        test_sets.append((seed, set(ss_summary[key]["test_students"])))
    
    print(f"\n[{ratio_name}]")
    for i in range(len(test_sets)):
        for j in range(i + 1, len(test_sets)):
            s1, set1 = test_sets[i]
            s2, set2 = test_sets[j]
            overlap = set1 & set2
            pct = len(overlap) / len(set1) * 100
            print(f"  Seed {s1} vs {s2}: 겹침 {len(overlap)}/{len(set1)}명 ({pct:.1f}%)")

=== 생성 파일 확인 ===

[question_data] 8개 파일
  Q1.xlsx: 265건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q2.xlsx: 265건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q3.xlsx: 265건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q4.xlsx: 265건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q5.xlsx: 255건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q6.xlsx: 255건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q7.xlsx: 255건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']
  Q8.xlsx: 255건, 컬럼=['id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized']

[student_splits] 30개 파일
  test_40_60_seed123.xlsx: 1280건
  test_40_60_seed42.xlsx: 1220건
  test_40_60_seed456.xlsx: 1220건
  test_50_50_seed123.xlsx: 1080건
  test_50_50_seed42.xlsx: 10